# Guide to RAG Evaluations

Retrieval-Augmented Generation (RAG) pipelines consist of two distinct systems working together: the **Retriever** (finding relevant context) and the **Generator** (the LLM synthesizing the answer).

Because these are separate processes, evaluating a RAG pipeline requires isolating and measuring both components independently. A failure in a RAG system usually falls into one of two categories:
1. The retriever failed to find the right information.
2. The generator hallucinated or ignored the retrieved information.

---

## 1. Retrieval Evaluations (Evaluating the "R")
Retrieval metrics measure the performance of your vector database and embedding models. They answer the question: *Did we fetch the correct context, and did we rank it highly?*

### A. Precision@K
Measures the proportion of retrieved documents in the top $K$ results that are actually relevant to the user's query. It evaluates the "purity" of your retrieved chunks.

* **Formula:** $$\text{Precision@K} = \frac{\text{Number of relevant documents in top } K}{K}$$
* **Explanation:** If you retrieve 5 chunks ($K=5$) and 3 of them contain the answer, your Precision@5 is 0.6 (60%).

### B. Recall@K
Measures the proportion of *all* actual relevant documents that were successfully captured in the top $K$ results.

* **Formula:** $$\text{Recall@K} = \frac{\text{Number of relevant documents in top } K}{\text{Total number of relevant documents in the entire database}}$$
* **Explanation:** If there are 10 total chunks in your knowledge base that contain parts of the answer, and your retriever only grabs 4 of them in its top $K$ results, your Recall@K is 0.4 (40%).

### C. Mean Reciprocal Rank (MRR)
Evaluates how far down the list the user has to look to find the *first* relevant document. It heavily penalizes systems that put the best document at the bottom of the retrieval list.

* **Formula:** $$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$
    *(Where $|Q|$ is the total number of queries, and $\text{rank}_i$ is the rank position of the first relevant document for the $i$-th query).*
* **Explanation:** If the first relevant chunk is the very top result (Rank 1), the score is $1/1 = 1.0$. If the first relevant chunk is the 4th result, the score is $1/4 = 0.25$. MRR averages these fractions across all test queries.

### D. Normalized Discounted Cumulative Gain (NDCG)
A highly sophisticated metric that measures the overall quality of the ranking. Unlike MRR which only cares about the *first* relevant document, NDCG evaluates the position of *all* relevant documents, assigning higher scores if highly relevant documents appear at the very top.

* **Formula:** $$\text{DCG}_p = \sum_{i=1}^{p} \frac{rel_i}{\log_2(i + 1)}$$
    $$\text{NDCG}_p = \frac{\text{DCG}_p}{\text{IDCG}_p}$$
    *(Where $rel_i$ is the relevance score of the document at position $i$, and IDCG is the Ideal DCG if the documents were sorted perfectly).*
* **Explanation:** It uses a logarithmic discount to penalize relevant documents that appear lower in the search results. An NDCG of 1.0 means the retriever returned every relevant document in the exact perfect order.

---

## 2. Generation Evaluations (Evaluating the "G")
Generation metrics evaluate the LLM's response. Because traditional NLP metrics like BLEU or ROUGE are terrible at understanding semantic meaning, modern RAG evaluation uses **"LLM-as-a-Judge"** frameworks (like RAGAS or TruLens) to score the output.

These metrics answer the question: *Did the LLM use the retrieved context correctly without hallucinating?*

### A. Faithfulness (Groundedness)
Measures whether the generated answer is strictly derived from the retrieved context. It is the primary defense against hallucinations.

* **Formula:** $$\text{Faithfulness} = \frac{\text{Number of claims in the generated answer that can be inferred from context}}{\text{Total number of claims in the generated answer}}$$
* **Explanation:** The evaluator LLM breaks the generated answer down into individual factual claims. It then checks the retrieved context to see if each claim is supported. If the model generated 4 claims, but 1 of them was a hallucination not found in the context, the Faithfulness score is 0.75.

### B. Answer Relevance
Measures how directly the generated answer addresses the user's original query. It penalizes evasive answers or answers that go on a tangent, even if they are factually correct.

* **Formula:** $$\text{Answer Relevance} = \frac{1}{N} \sum_{i=1}^{N} \cos(E(q), E(q_i))$$
    *(Where $E(q)$ is the embedding of the original query, and $E(q_i)$ are embeddings of questions reverse-engineered from the generated answer).*
* **Explanation:** The evaluator LLM reads the final answer and is asked to *guess* what the original question was. If the generated question closely matches the user's actual question (measured by cosine similarity of their embeddings), the answer is highly relevant.

### C. Context Precision (RAGAS specific)
Evaluates whether all of the ground-truth relevant items present in the contexts are ranked higher or lower. This is a bridge metric checking if the generator was handed a clean, highly relevant prompt.

* **Formula:** $$\text{Context Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\text{Total number of relevant items in top K}}$$
    *(Where $v_k$ is an indicator function being 1 if the item at rank $k$ is relevant, and 0 otherwise).*
* **Explanation:** It acts as a strict check to ensure that the LLM isn't being forced to read through a massive amount of "noise" (irrelevant context) before finding the "signal" (relevant context) at the very bottom of the prompt.